# 2-1절 연습 문제 풀이

이 노트북은 2-1절 연습 문제(2-1 ~ 2-3)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `code_examples/ch02/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
# 환경 설정 - 시드 고정 (예제 노트북과 같은 SEED)
import random

import numpy as np
import torch

SEED = 2
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 연습 문제 2-1

> [그림 2-2]의 퍼셉트론에서 가중치가 각각 0.5와 -0.5, 편향이 0.1일 때,
> 입력 (1, 0)과 (0, 1)에 대한 뉴런의 가중합 z를 계산해 보자.

### 손으로 푼 풀이

뉴런의 가중합은 z = x1w1 + x2w2 + b다.

- 입력 (1, 0): z = 1 × 0.5 + 0 × (-0.5) + 0.1 = **0.6**
- 입력 (0, 1): z = 0 × 0.5 + 1 × (-0.5) + 0.1 = **-0.4**

두 입력의 부호가 갈리는 이유는 두 가중치의 부호가 다르기 때문이다.
첫 번째 입력은 출력을 키우는 쪽으로, 두 번째 입력은 줄이는 쪽으로 작용한다.

In [2]:
weights = torch.tensor([[0.5], [-0.5]])   # (2, 1) 형태
bias = torch.tensor([0.1])
X = torch.tensor([[1., 0.], [0., 1.]])

# 본문 코드 2-3과 같은 방식으로 행렬곱을 사용해 한 번에 계산
Z = X @ weights + bias
for x, z in zip(X.tolist(), Z.flatten().tolist()):
    print(f'입력 {tuple(x)}의 가중합 z: {z:.1f}')

입력 (1.0, 0.0)의 가중합 z: 0.6
입력 (0.0, 1.0)의 가중합 z: -0.4


### 문제 검토

- **적절성: 적합.** 본문이 설명한 가중합 식을 손으로 한 번 계산하게 하는 확인 문제다.
  두 가중치의 부호를 다르게 준 덕분에 z의 부호가 갈려, 다음 문제 2-2에서 시그모이드를 통과시켰을 때
  0.5를 기준으로 양쪽으로 나뉘는 것까지 자연스럽게 이어진다. 숫자 선택이 좋다.
- **[검토] 계산 결과를 어디에 쓰는지 밝혀 두면 좋다.** 이 문제의 z 값은 2-2에서 그대로 쓰인다.
  지문에 그 연결을 한마디 적어 두면 독자가 답을 적어 두고 넘어간다.

## 연습 문제 2-2

> [그림 2-2]의 퍼셉트론에서 시그모이드 활성화 함수를 거친 출력값 y의 값을 반환하는 파이썬 함수를 작성한 후,
> [연습 문제 2-1]의 조건을 사용해 퍼셉트론의 출력값을 구해 보자.

In [3]:
def perceptron(X, weights, bias):
    """가중합을 구한 뒤 시그모이드 활성화 함수를 거친 출력을 반환한다."""
    Z = X @ weights + bias
    return 1 / (1 + torch.exp(-Z))      # 시그모이드 함수

Y = perceptron(X, weights, bias)
for x, y in zip(X.tolist(), Y.flatten().tolist()):
    label = 1 if y >= 0.5 else 0
    print(f'입력 {tuple(x)} -> 출력 {y:.4f} (0.5 기준 분류: {label})')

입력 (1.0, 0.0) -> 출력 0.6457 (0.5 기준 분류: 1)
입력 (0.0, 1.0) -> 출력 0.4013 (0.5 기준 분류: 0)


### 풀이 해설

z = 0.6은 시그모이드를 거쳐 0.6457이 되고, z = -0.4는 0.4013이 된다.
시그모이드 함수는 z = 0에서 0.5를 지나므로, **z의 부호가 곧 0.5를 기준으로 한 분류 결과**를 결정한다.
본문이 결정 경계를 z = 0으로 잡은 이유(각주 6)를 숫자로 확인할 수 있다.

`torch.sigmoid(Z)`로 바꿔 써도 결과는 같다.

### 문제 검토

- **적절성: 적합.** 2-1과 짝을 이뤄 '가중합 → 활성화 → 분류'라는 퍼셉트론의 흐름을 한 번에 훑게 한다.
  본문 코드 2-3을 그대로 따라 쓰면 풀리므로 난도도 알맞다.
- **[검토] '출력값 y의 값을 반환하는'.** '값'이 겹친다. '출력값 y를 반환하는'으로 줄이면 읽기 편하다.
- **[검토] 무엇을 확인해야 하는지 덧붙이면 좋다.** 이 문제의 핵심은 두 출력이 0.5를 사이에 두고 갈린다는 점인데,
  지문은 '출력값을 구해 보자'에서 끝난다. 한 구절이면 관찰 대상이 분명해진다.

**윤문안**

> **2-2**. [그림 2-2]의 퍼셉트론에서 시그모이드 활성화 함수를 거친 출력값 y를 반환하는 파이썬 함수를 작성한 후,
> [연습 문제 2-1]의 조건을 사용해 퍼셉트론의 출력값을 구해 보자.
> 두 출력값을 0.5와 비교해 보고, 그 결과가 [연습 문제 2-1]에서 구한 z의 부호와 어떻게 대응되는지도 확인해 보자.

## 연습 문제 2-3

> 퍼셉트론의 결정 경계는 가중치와 편향에 따라 달라진다.
> 다음 세 가지 경우에서 결정 경계의 기울기와 x2축 절편을 각각 구하고 결정 경계가 어떻게 달라지는지 설명해 보자.
>
> - w1=1, w2=1, b=0
> - w1=2, w2=1, b=0
> - w1=1, w2=1, b=1

### 손으로 푼 풀이

본문 p12의 식에 따라 결정 경계는 기울기가 -w1/w2, x2축 절편이 -b/w2인 직선이다.

| 경우 | 가중치, 편향 | 기울기 | x2축 절편 |
|---|---|---|---|
| ① | w1=1, w2=1, b=0 | -1 | 0 |
| ② | w1=2, w2=1, b=0 | -2 | 0 |
| ③ | w1=1, w2=1, b=1 | -1 | -1 |

①과 ②를 비교하면 **가중치의 비율이 기울기를 바꾼다.** w1이 2배가 되자 경계가 더 가파르게 섰다.
x1의 영향력이 커졌으므로, x1이 조금만 변해도 분류 결과가 바뀌는 방향으로 경계가 기운 것이다.
①과 ③을 비교하면 **편향이 경계를 평행 이동시킨다.** 기울기는 그대로인 채 경계가 아래로 1만큼 내려갔다.
편향이 커질수록 경계가 원점에서 멀어지며, 출력이 1이 되는 영역이 넓어진다.

In [4]:
CASES = [(1., 1., 0.), (2., 1., 0.), (1., 1., 1.)]

print(f'{"w1":>4}{"w2":>4}{"b":>4} | {"기울기":>8} {"x2축 절편":>10}')
print('-' * 34)
for w1, w2, b in CASES:
    slope = -w1 / w2
    intercept = -b / w2
    print(f'{w1:4.0f}{w2:4.0f}{b:4.0f} | {slope:8.1f} {intercept:10.1f}')

  w1  w2   b |      기울기     x2축 절편
----------------------------------
   1   1   0 |     -1.0       -0.0
   2   1   0 |     -2.0       -0.0
   1   1   1 |     -1.0       -1.0


### 문제 검토

- **적절성: 적합.** 세 경우를 ①②는 가중치만, ①③은 편향만 바꾸도록 설계해 두 파라미터의 역할이
  따로 드러나게 했다. 변수를 하나씩만 바꾸는 좋은 설계다.
- **[검토] 코드가 필요 없는 문제인데 절의 다른 문제와 성격이 다르다.** 2-1, 2-2는 계산과 구현이고 2-3은
  식 정리다. 문제 자체는 좋으므로 그대로 두되, 답이 맞았는지 확인할 수 있게 "세 경계를 좌표평면에 그려 보자"를
  덧붙이면 2-2절의 [그림 2-6], [그림 2-7]과 자연스럽게 이어진다.
- **[확인 요청] 결정 경계 식이 나오는 위치.** 이 문제는 2-1절 끝에 있는데, 기울기 -w1/w2와 절편 -b/w2를
  함께 제시하는 식은 **2-2절 p12**에 처음 나온다. 2-1절 p4에는 "두 가중치의 비율(-w1/w2)은 기울기를,
  편향은 위치를 정한다"까지만 있어서, 독자가 x2축 절편을 구하려면 식을 스스로 유도해야 한다.
  의도한 난도라면 그대로 두어도 되지만, 그렇지 않다면 절편까지 묻는 부분을 2-2절 연습 문제로 옮기거나
  지문에 유도 힌트를 붙이는 편이 좋다.